In [ ]:
!pip install eyepop==3.12.0
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 44.9 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [ ]:
GEMINI_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your API KEY: ··········


In [ ]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


roof_classify_prompt = """
Determine the primary roof condition in the image and assign exactly one label: ['No_Damage', 'Missing_Shingles', 'Hail_Impact', 'Ponding'].

Choose 'No_Damage' only if the image's main focus is a roof that appears intact, with no clearly visible missing shingles, hail impact marks, or standing water. Normal roof texture, mild dirt, leaves, roof vents, shadows, age discoloration, or lighting variation should not be classified as damage.

Choose 'Missing_Shingles' only if the image's main focus shows a shingle roof with one or more clearly missing, lifted, torn, or displaced shingles. This includes exposed underlayment, exposed decking, bare roof patches, uneven shingle rows, or visible gaps where shingles should be.

Choose 'Hail_Impact' only if the image's main focus shows visible hail-related surface damage. This includes circular impact marks, dents, bruised shingles, granule loss spots, pitting, or repeated small round damage patterns across the roof surface. Do not choose this label for normal shingle texture, dirt, shadows, glare, or general color variation.

Choose 'Ponding' only if the image's main focus shows standing water accumulated on a flat or low-slope roof. This includes visible puddles, reflective water pools, water collected around drains, or large wet areas where water has not drained. Do not choose this label for a roof that is merely damp without obvious pooled water.

If multiple roof conditions are visible, choose the most visually dominant roof damage type. If no clear damage is visible, choose 'No_Damage'.

Return only the single best-fitting label.
"""


roof_describe_prompt = """
You are a roof damage assessment assistant. Analyze the rooftop image and describe any visible roof damage.

Return only valid JSON. Do not include markdown or extra commentary.

Use this exact JSON structure:

{
  "roof_type": null,
  "damage_detected": null,
  "damage_type": null,
  "damage_location": null,
  "severity": null,
  "visual_evidence": null,
  "recommended_next_step": null
}

Instructions:
- "roof_type" should describe the visible roof material or roof style, such as asphalt shingle roof, tile roof, metal roof, flat membrane roof, or unknown.
- "damage_detected" should be true if clear roof damage is visible, otherwise false.
- "damage_type" should be one of: "missing_shingles", "hail_impact", "ponding", "no_visible_damage", or "unknown".
- "damage_location" should describe where the damage appears in the image, such as center roof section, lower left slope, around roof vent, near drain, or unknown.
- "severity" should be one of: "none", "minor", "moderate", "severe", or "unknown".
- "visual_evidence" should briefly explain the visible clues supporting the assessment.
- "recommended_next_step" should briefly state what a property owner, inspector, or roofing professional should do next.

Do not guess damage that is not clearly visible. If the image does not clearly show a roof, set "damage_type" to "unknown".
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.image-classify.roof-damage",
        description="Classify rooftop images by roof damage type",
        worker_release="qwen3-instruct",
        text_prompt=roof_classify_prompt,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=10,
            image_size=640
        ),
        is_public=False
    ),
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.roof-damage-report",
        description="Describe roof damage type, location, severity, and recommended next step",
        worker_release="qwen3-instruct",
        text_prompt=roof_describe_prompt,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=250,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))

        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )

        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )

        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )

        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a398323b6e7d8a8000308249555d62 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.image-classify.roof-damage', tag='1.0.1'), AbilityAliasEntry(alias='datasciencealliance-org.image-classify.roof-damage', tag='latest')]
created ability 06a398324f13768280000e09cf03356f with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.roof-damage-report', tag='1.0.1'), AbilityAliasEntry(alias='datasciencealliance-org.describe.roof-damage-report', tag='latest')]


### Evalulate on a Single Image

In [ ]:
from google import genai
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


sample_img_path = Path("/content/sample_img.jpg")

# Run roof damage classification ability

classification_pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.image-classify.roof-damage:latest"
    )
])

classification_results = []

print("=== RAW CLASSIFICATION RESULT ===")

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(classification_pop)
    job = endpoint.upload(sample_img_path)

    while result := job.predict():
        classification_results.append(result)
        print(json.dumps(result, indent=2))


# Run roof damage describe/report ability

describe_pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.roof-damage-report:latest"
    )
])

description_results = []

print("\n=== RAW DESCRIPTION RESULT ===")

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(describe_pop)
    job = endpoint.upload(sample_img_path)

    while result := job.predict():
        description_results.append(result)
        print(json.dumps(result, indent=2))


# Use Gemini to synthesize a clean final roof damage report

all_results_string = json.dumps(
    {
        "classification_results": classification_results,
        "description_results": description_results
    },
    indent=2
)

prompt = f"""
You are a roof damage assessment assistant.

I am providing you with raw AI outputs from two EyePop abilities:
1. A roof damage classification ability.
2. A roof damage description ability.

Your job is to synthesize the outputs into one clean final roof damage report.

Use only the information provided in the raw outputs. Do not invent damage that is not supported by the visual analysis.

Provide the final report in this exact structure:

* Primary Classification: [No_Damage, Missing_Shingles, Hail_Impact, or Ponding]
* Roof Type: [roof material or roof style]
* Damage Detected: [true or false]
* Damage Type: [missing shingles, hail impact, ponding, no visible damage, or unknown]
* Damage Location: [where the damage appears]
* Severity: [none, minor, moderate, severe, or unknown]
* Visual Evidence: [brief explanation of visible clues]
* Recommended Next Step: [brief practical recommendation]

Here is the raw EyePop output:
{all_results_string}
"""

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("\n=== FINAL ROOF DAMAGE REPORT ===")
print(response.text)

print("\nDone")

=== RAW CLASSIFICATION RESULT ===
{
  "compute_units": 0.1,
  "seconds": 0,
  "source_height": 559,
  "source_id": "e9638248-6e6a-11f1-8bca-d62626e87042",
  "source_width": 1024,
  "system_timestamp": 1782154090845330000,
  "texts": [
    {
      "id": 1,
      "text": "Hail_Impact"
    }
  ],
  "timestamp": 0
}

=== RAW DESCRIPTION RESULT ===
{
  "compute_units": 0.1,
  "seconds": 0,
  "source_height": 559,
  "source_id": "eb482420-6e6a-11f1-8bca-d62626e87042",
  "source_width": 1024,
  "system_timestamp": 1782154099023035000,
  "texts": [
    {
      "id": 1,
      "text": "{\n  \"roof_type\": \"asphalt shingle roof\",\n  \"damage_detected\": true,\n  \"damage_type\": \"hail_impact\",\n  \"damage_location\": \"across visible roof slope, concentrated in center and lower sections\",\n  \"severity\": \"moderate\",\n  \"visual_evidence\": \"Multiple circular holes and dented areas on shingles consistent with hail impact; shingles show granule loss and deformation\",\n  \"recommended_next

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}